In [2]:
import os

# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 1.定义工具

我们先通过一个例子来看看工具的作用。

假设我们需要一个能计算数学题的智能体。

但是，模型本身不擅长处理数学问题，如果涉及到精确计算，它往往无法给出准确答案


In [3]:
import os
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider="openai",
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("SILICONFLOW_API_KEY"),
)

agent = create_agent(
    model=model,
    system_prompt="你是一个算术奇才。"
)

In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content="467的平方根是多少?")

for token, metadata in agent.stream(
    {"messages": [question]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)



467的平方根是一个无理数，无法用分数精确表示。通过牛顿迭代法或近似计算，可以得到其近似值。以下是详细步骤：

1. **确定范围**  
   21² = 441，22² = 484，因此√467位于21和22之间。

2. **初始近似值**  
   观察到21.6² = 466.56（与467非常接近），因此选择初始猜测值为21.6。

3. **牛顿迭代法计算**  
   使用公式 $ x_{n+1} = \frac{x_n + \frac{S}{x_n}}{2} $，其中 $ S = 467 $：  
   - 第一次迭代：  
     $ x_0 = 21.6 $  
     $ x_1 = \frac{21.6 + \frac{467}{21.6}}{2} \approx \frac{21.6 + 21.6185}{2} \approx 21.61425 $  
   - 第二次迭代：  
     $ x_1 ≈ 21.61425 $  
     $ x_2 = \frac{21.61425 + \frac{467}{21.61425}}{2} \approx \frac{21.61425 + 21.61018}{2} \approx 21.6122 $

4. **进一步验证**  
   计算 $ 21.610185^2 \approx 467.000 $，误差极小，因此四舍五入后：  
   - **保留三位小数**：√467 ≈ **21.610**  
   - **保留四位小数**：√467 ≈ **21.6102**  
   - **更精确的小数**：21.610185...

**最终答案**：  
467的平方根约为 **21.61**（保留两位小数），更精确的值为 **21.6102**（四位小数）。通常使用计算器可直接得到更高精度的结果。

所以，通常涉及到精确的数学计算，我们都会通过定义工具用传统代码来计算，然后把工具交给模型去使用。


In [5]:
# 定义工具
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [6]:
# 通过装饰器定义工具名称
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [7]:
# 通过装饰器定义工具作用描述
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [8]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal


# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


工具调用方式与普通函数调用方式一致。


In [9]:
tool1.invoke({"x": 467})

21.61018278497431

In [10]:
get_weather.invoke({"location": "杭州", "include_forecast": True})

'Current weather in 杭州: 22 degrees C\nNext 5 days: Sunny'

# 2.添加工具到智能体


In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[tool1, get_weather],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

In [12]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="467的平方根是多少?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)




21.61018278497431

467的平方根是约21.61。

In [13]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="北京和杭州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)



Current weather in 北京: 22 degrees C
Next 5 days: SunnyCurrent weather in 杭州: 22 degrees C
Next 5 days: Sunny

北京和杭州的天气情况如下：

**北京**  
当前温度：22°C  
接下来5天：晴天  

**杭州**  
当前温度：22°C  
接下来5天：晴天  

两地天气均以晴朗为主，温度适宜，适合户外活动。需要进一步帮助吗？

如果采用stream的updates模式，可以看到Agent执行的每一个步骤：

In [14]:
for chunk in agent.stream(
    {"messages": [HumanMessage(content="467、529的平方根是多少?")]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print()


step: model
content: [{'type': 'tool_call', 'name': 'square_root', 'args': {'x': 467}, 'id': '019e1af81b95a8c346213dd87a8f6fdd'}, {'type': 'tool_call', 'name': 'square_root', 'args': {'x': 529}, 'id': '019e1af81b95a8c346213dd87a8f6fde'}]

step: tools
content: [{'type': 'text', 'text': '21.61018278497431'}]

step: tools
content: [{'type': 'text', 'text': '23.0'}]

step: model
content: [{'type': 'text', 'text': '467的平方根约为21.61，529的平方根是23。'}]



# 3.预定义Tool

LangChain中提供了很多预定义的Tool，方便我们使用。例如：


In [15]:
# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch

tool = TavilySearch(
    max_results=5,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [16]:
tool.invoke("蒸蚌是什么梗？")

{'query': '蒸蚌是什么梗？',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://zh.hinative.com/questions/15904858',
   'title': '"我蒸蚌"是什么意思？ - HiNative',
   'content': '我蒸蚌蒸蚌是「真棒」的意思因為“蒸蚌” 跟“真棒” 的音一模一樣，所以有一些人在寫「真棒」的時候，會故意用蒸蚌兩字，然後只是因為這樣子看起來很有趣 的',
   'score': 0.9998104,
   'raw_content': None},
  {'url': 'https://www.facebook.com/61557545415108/posts/%E8%92%B8%E8%9A%8Cmemes-fun-%E8%BF%B7%E5%9B%A0-%E6%90%9E%E7%AC%91/122172017300251513/',
   'title': '蒸蚌！ #memes #fun #迷因#搞笑',
   'content': 'Abracadabra是一個著名的咒文，作為「魔語 」在進行魔術表演時使用。歷史上曾經認為， 這個咒文刻在護身符時能有治癒的能力。來自 維基百科 這個送去',
   'score': 0.9995122,
   'raw_content': None},
  {'url': 'https://www.sina.cn/news/detail/5251386149438121.html',
   'title': '蒸蚌梗稀缺性引关注_新浪新闻',
   'content': '原来这一声单纯的蒸蚌是这么稀缺的东西。 这才是这个梗能爆出来的真正原因吧。 原来我都没往这边想，这种视频看多了差点脱口而出给别人喊蒸蚌。',
   'score': 0.99946004,
   'raw_content': None},
  {'url': 'https://www.facebook.com/61586315125667/posts/%E9%9A%A8%E4%BE%BF%E6%8C%87%E4%B8%80%E6%8C%87%E8%81%BD%E

In [17]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model=model,
    tools=[tool],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

In [18]:
# 调用工具
for chunk in agent.stream(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print()

step: model
content: [{'type': 'tool_call', 'name': 'tavily_search', 'args': {'query': '蒸蚌是什么梗', 'search_depth': 'advanced', 'topic': 'general'}, 'id': '019e1af8cd19cc1e64734eb0898ad1c0'}]

step: tools
content: [{'type': 'text', 'text': '{"query": "蒸蚌是什么梗", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.adquan.com/article/359318", "title": "萝卜纸巾分不清的“蒸蚌小猫”，拿下首个全球代言！ - 广告门", "content": "“动物营销”的信任优势：\\n\\n相比于人类明星可能出现的“人设塌房”风险，动物IP的形象更加纯粹、稳定且安全。\\n\\n它们的“人设”几乎不会崩塌，且天生具备亲近感和治愈力，能有效降低消费者对广告的抵触心理，提升品牌好感度。德祐结合公益的举措，更是将这种好感度转化为品牌美誉度。\\n\\n谐音梗与“梗文化”的胜利：\\n\\n“蒸蚌”作为“真棒”的谐音，本身就是一个极具记忆点和传播力的符号。它代表了一种幽默、自我调侃式的鼓励文化。\\n\\n品牌使用这个梗，瞬间就与年轻用户建立了“自己人”的沟通语境，完成了从“高高在上”到“一起玩梗”的身份转变。\\n\\n归根结底，“大开门”和“萝卜纸巾”的爆火，是一场关于“鼓励”的全民共谋。\\n\\n我们在这个简单游戏里，投射了自己对“被肯定”的渴望。而品牌们的成功“吻猫”，则证明了在新的营销时代，比追逐流量更重要的，是理解流量背后涌动的情结。\\n\\n当一只猫因为“分不清萝卜和纸巾”而被全世界鼓励“蒸蚌”时，聪明的品牌要做的，不是解释规则，而是和大家一起，为这份珍贵的、单纯的快乐鼓掌，然后，恰到好处地递上一包联名湿厕纸，或是一台能清洁生活的机器。\\n\\n虽然生活的考题常常没有标准答案，但一份“蒸蚌”式的肯定，永远稀缺。\\n\\n萝卜还是纸巾？这不重要。重要的是，

In [22]:
from langchain.tools import tool
from langchain_tavily import TavilySearch

tavily = TavilySearch(
    max_results=5,
    topic="general"
)

@tool
def web_search(query: str):
    """Search the web for information"""
    return tavily.invoke({"query": query})


In [24]:
from pydantic import BaseModel, Field
import os
from langchain.chat_models import init_chat_model



# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo (BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

model = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    model_provider="openai",
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("SILICONFLOW_API_KEY"),
)

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
    response_format=AnswerInfo
)

# 调用agent
response = agent.invoke(
    {"messages": [HumanMessage(content="蒸蚌是什么梗？")]},
)

# 获取结构化输出
print(response['structured_response'])

answer='‘蒸蚌’是一个源自网络的谐音梗，起源于一只名叫‘大开门’的三花猫。这只猫因主人训练它分辨萝卜和纸巾而走红，主人在它指错答案时总说‘蒸蚌’（即‘真棒’），用一种幽默、自我调侃的方式传递鼓励文化。这个梗随后被广泛传播，衍生出各种二创形式，如让宠物或人类参与类似挑战，甚至开发成表情包、网页小游戏等，成为一种情绪价值的表达方式。它代表了现代年轻人的一种情感投射：在面对不确定性时，渴望被无条件接纳和鼓励。' reference=[Reference(title='萝卜纸巾分不清的“蒸蚌小猫”，拿下首个全球代言！ - 广告门', url='https://www.adquan.com/article/359318'), Reference(title='米老鼠蒸蚌意义 - 抖音', url='https://www.douyin.com/shipin/7587218870990784546'), Reference(title='蒸蚌小猫喜提友望全球代言，品牌金主纷纷吻了上来 - 4A广告网', url='https://www.4anet.com/p/01km5gcp98wttejc3129evhhgm'), Reference(title='一只笨猫学习辨认萝卜和纸巾，竟成了近期最火的二创赛道？ - 虎嗅', url='https://www.huxiu.com/article/4820519.html'), Reference(title='爆火的年末热梗，藏着年轻人的精神密码', url='https://www.woshipm.com/it/6319040.html')]
